# 11 — H2b: Operating Performance Panel Regressions

**Goal:** Test whether lagged employee sentiment predicts next-year operating
performance (ROA, operating margin, sales growth) using firm and year fixed-effects
panel regressions.

**Input:** `data/panel_yearly.parquet` (6,778 firm-years)

**Output:** `output/table_operating_performance.csv`

**Specification (pre-committed):**
- Three dependent variables: ROA, operating margin, sales growth (year t)
- Key regressor: employee sentiment (year t−1, already lagged in the panel)
- Controls: log assets, leverage
- Firm fixed effects (EntityEffects) + year fixed effects (TimeEffects)
- Standard errors clustered by firm
- Outcome variables winsorised at 1%/99% to limit the influence of extreme values

**Interpretation:** A positive, significant coefficient on lagged sentiment indicates
that within-firm increases in employee sentiment predict subsequent improvements in
operating performance.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from linearmodels.panel import PanelOLS
from scipy.stats import mstats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"
OUTPUT = Path.home() / "thesis" / "output"

panel = pd.read_parquet(DATA_PROCESSED / "panel_yearly.parquet")
print(f"Panel: {len(panel):,} firm-years, {panel['gvkey'].nunique()} firms")
print(f"Year range: {panel['fyear'].min()} to {panel['fyear'].max()}")

# --- NaN-safe winsorization fix (2026-06-19): mstats.winsorize silently clips
# missing values to the cap, fabricating firm-years. Preserve NaN instead so the
# regression's dropna handles them. ---
import pandas as _pd
_orig_winsorize = mstats.winsorize
def _winsorize_nan_safe(a, *args, **kwargs):
    s = _pd.Series(a).astype(float); out = s.copy(); m = s.notna()
    out[m] = _orig_winsorize(s[m].to_numpy(), *args, **kwargs)
    return out.to_numpy()
mstats.winsorize = _winsorize_nan_safe


Panel: 5,434 firm-years, 635 firms
Year range: 2013 to 2024


In [2]:
# Winsorise the outcome variables at 1%/99% to limit extreme-value influence
# (recall: operating margin had a -854% min, sales growth a +2199% max)
for var in ["roa", "operating_margin", "sales_growth"]:
    panel[f"{var}_w"] = mstats.winsorize(
        panel[var].astype(float), limits=[0.01, 0.01]
    )

# Also winsorise controls lightly (leverage had a max of 4.35)
panel["leverage_w"] = mstats.winsorize(panel["leverage"].astype(float), limits=[0.01, 0.01])

# Restrict to the 2013-2023 headline window AFTER winsorizing (caps come from the full panel; per JJ's directed window). 2024 enters only as the footnote below.
panel = panel[(panel["fyear"] >= 2013) & (panel["fyear"] <= 2023)].copy()

print("Before/after winsorisation (outcome variables):")
for var in ["roa", "operating_margin", "sales_growth"]:
    print(f"  {var:18s} raw range: [{panel[var].min():.2f}, {panel[var].max():.2f}]  "
          f"winsorised: [{panel[f'{var}_w'].min():.2f}, {panel[f'{var}_w'].max():.2f}]")
print()

# Set up the panel index: linearmodels needs a MultiIndex of (entity, time)
panel_idx = panel.set_index(["gvkey", "fyear"])

# Drop rows missing the key regressor or any control
panel_idx = panel_idx.dropna(subset=["sentiment_overall_lag", "log_at", "leverage_w"])
print(f"Panel ready for regression: {len(panel_idx):,} firm-years")

Before/after winsorisation (outcome variables):
  roa                raw range: [-0.85, 0.51]  winsorised: [-0.15, 0.28]
  operating_margin   raw range: [-2.87, 0.82]  winsorised: [-0.04, 0.66]
  sales_growth       raw range: [-0.84, 21.99]  winsorised: [-0.42, 0.78]

Panel ready for regression: 4,956 firm-years


In [3]:
results = {}
outcomes = {
    "roa_w": "Return on assets",
    "operating_margin_w": "Operating margin",
    "sales_growth_w": "Sales growth",
}

for outcome_var, outcome_label in outcomes.items():
    # Build the regression: outcome ~ lagged sentiment + controls + firm FE + year FE
    formula = (
        f"{outcome_var} ~ sentiment_overall_lag + log_at + leverage_w "
        f"+ EntityEffects + TimeEffects"
    )
    model = PanelOLS.from_formula(formula, data=panel_idx, drop_absorbed=True)
    res = model.fit(cov_type="clustered", cluster_entity=True)
    results[outcome_var] = res

    # Pull the key coefficient
    coef = res.params["sentiment_overall_lag"]
    tstat = res.tstats["sentiment_overall_lag"]
    pval = res.pvalues["sentiment_overall_lag"]

    print(f"{'='*60}")
    print(f"OUTCOME: {outcome_label}")
    print(f"{'='*60}")
    print(f"  Sentiment coefficient: {coef:.5f}")
    print(f"  t-statistic:           {tstat:.3f}")
    print(f"  p-value:               {pval:.4f}")
    print(f"  Significant at 5%?     {'YES' if pval < 0.05 else 'No'}")
    print(f"  Within R-squared:      {res.rsquared_within:.4f}")
    print(f"  N observations:        {res.nobs:,}")
    print(f"  N firms:               {res.entity_info.total}")
    print()

OUTCOME: Return on assets
  Sentiment coefficient: 0.00379
  t-statistic:           1.680
  p-value:               0.0931
  Significant at 5%?     No
  Within R-squared:      0.0325
  N observations:        4,956
  N firms:               625.0

OUTCOME: Operating margin
  Sentiment coefficient: 0.00095
  t-statistic:           0.220
  p-value:               0.8258
  Significant at 5%?     No
  Within R-squared:      0.0401
  N observations:        4,725
  N firms:               596.0

OUTCOME: Sales growth
  Sentiment coefficient: 0.02110
  t-statistic:           1.796
  p-value:               0.0726
  Significant at 5%?     No
  Within R-squared:      0.0164
  N observations:        4,954
  N firms:               625.0



/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [4]:
# Assemble a results table
summary_rows = []
for outcome_var, outcome_label in outcomes.items():
    res = results[outcome_var]
    summary_rows.append({
        "outcome": outcome_label,
        "sentiment_coef": res.params["sentiment_overall_lag"],
        "t_stat": res.tstats["sentiment_overall_lag"],
        "p_value": res.pvalues["sentiment_overall_lag"],
        "within_r2": res.rsquared_within,
        "n_obs": int(res.nobs),
        "n_firms": int(res.entity_info.total),
    })

results_table = pd.DataFrame(summary_rows)
print(results_table.round(4))
results_table.round(4).to_csv(OUTPUT / "table_operating_performance.csv", index=False)
print(f"\nSaved to {OUTPUT / 'table_operating_performance.csv'}")

            outcome  sentiment_coef  t_stat  p_value  within_r2  n_obs  n_firms
0  Return on assets          0.0038  1.6797   0.0931     0.0325   4956      625
1  Operating margin          0.0009  0.2201   0.8258     0.0401   4725      596
2      Sales growth          0.0211  1.7955   0.0726     0.0164   4954      625

Saved to /Users/<wrds-username>/thesis/output/table_operating_performance.csv


In [5]:
# Robustness footnote: H2b results materially unchanged when 2024 is included (full 2013-2024 sample).
panel_full = pd.read_parquet(DATA_PROCESSED / "panel_yearly.parquet")
for var in ["roa","operating_margin","sales_growth"]:
    panel_full[f"{var}_w"] = mstats.winsorize(panel_full[var].astype(float), limits=[0.01,0.01])
panel_full["leverage_w"] = mstats.winsorize(panel_full["leverage"].astype(float), limits=[0.01,0.01])
pf = panel_full.set_index(["gvkey","fyear"]).dropna(subset=["sentiment_overall_lag","log_at","leverage_w"])
print("Footnote -- H2b including 2024 (full 2013-2024 sample):")
for o in ["roa_w","operating_margin_w","sales_growth_w"]:
    r = PanelOLS.from_formula(f"{o} ~ sentiment_overall_lag + log_at + leverage_w + EntityEffects + TimeEffects", pf, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)
    print(f"  {o:20s} coef={r.params['sentiment_overall_lag']:.5f}  t={r.tstats['sentiment_overall_lag']:.2f}  p={r.pvalues['sentiment_overall_lag']:.4f}")


Footnote -- H2b including 2024 (full 2013-2024 sample):
  roa_w                coef=0.00459  t=2.00  p=0.0461
  operating_margin_w   coef=0.00090  t=0.22  p=0.8288
  sales_growth_w       coef=0.02314  t=2.10  p=0.0359


/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
